In [ ]:
#@title Setup — run once, then go to the next cell { display-mode: "form" }
!pip install -q requests

import csv, io, re, warnings
from dataclasses import dataclass, field
import requests

# ── fetch ──────────────────────────────────────────────────────────────────────

def fetch_csv(url):
    m = re.search(r'/spreadsheets/d/([a-zA-Z0-9_-]+)', url)
    if not m:
        raise ValueError(f'Not a valid Google Sheets URL: {url}')
    sheet_id = m.group(1)
    gid_m = re.search(r'[#&?]gid=(\d+)', url)
    export = f'https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv'
    if gid_m:
        export += f'&gid={gid_m.group(1)}'
    r = requests.get(export)
    r.raise_for_status()
    return r.text

# ── parse ──────────────────────────────────────────────────────────────────────

_KEYWORDS_AS_REST = {'levada', 'virada'}
_BAR_RANGE_RE = re.compile(r'^\d+\s*-\s*\d+$')

@dataclass
class Break:
    name: str
    tracks: dict = field(default_factory=dict)

def _normalize_note(cell):
    s = cell.strip()
    return '0' if (not s or s.lower() in _KEYWORDS_AS_REST) else s

def _finalize_break(brk):
    if not brk.tracks:
        return
    max_len = max(len(v) for v in brk.tracks.values())
    for notes in brk.tracks.values():
        notes.extend(['0'] * (max_len - len(notes)))

def parse_sheet(csv_text):
    reader = csv.reader(io.StringIO(csv_text))
    breaks, current_break, in_bar_group, bar_group_offset = [], None, False, 0
    for raw_row in reader:
        row = raw_row + [''] * max(0, 65 - len(raw_row))
        col0 = row[0].strip()
        note_cells = [row[i].strip() for i in range(1, 65)]
        if not col0 and all(c == '' for c in note_cells):
            if in_bar_group:
                bar_group_offset += 64
            in_bar_group = False
            continue
        if _BAR_RANGE_RE.match(col0):
            if in_bar_group:
                bar_group_offset += 64
            in_bar_group = True
            if current_break is None:
                current_break = Break(name='')
                breaks.append(current_break)
            continue
        col1 = note_cells[0] if note_cells else ''
        rest_empty = all(c == '' for c in note_cells[1:])
        header_name = col0 or col1
        if header_name and not in_bar_group and (not col0 or not col1) and rest_empty:
            if current_break is not None:
                _finalize_break(current_break)
            current_break = Break(name=header_name)
            breaks.append(current_break)
            bar_group_offset = 0
            continue
        if col0 and in_bar_group and current_break is not None:
            notes = [_normalize_note(c) for c in note_cells]
            if col0 not in current_break.tracks:
                current_break.tracks[col0] = ['0'] * bar_group_offset
            current_break.tracks[col0].extend(notes)
    if current_break is not None:
        _finalize_break(current_break)
    return [b for b in breaks if b.tracks]

# ── mapping ────────────────────────────────────────────────────────────────────

@dataclass
class MappedTrack:
    instrument_id: str
    notes: list

_AGOGO    = {'L':'1','H':'2'}
_CHOCALHO = {'X':'1'}
_TAMBORIM = {'X':'1'}
_REPIQUE  = {'X':'1','x':'2','/':'3','K':'4','W':'5','O':'6','S':'7'}
_CAIXA    = {'X':'1','x':'2','W':'3','/':'4'}
_TIMBAU   = {'S':'2','O':'3'}
_HIGH_SURDO = {'X':'1','D':'2'}
_LOW_SURDO  = {'1':'1','O':'1','D':'2'}
_MID_SURDO  = {'2':'1','O':'1','D':'2'}

def _classify(name):
    n = name.lower()
    if 'surdo' in n:
        return 'high_surdo' if ('mor' in n or '3' in n) else 'surdo_split'
    for kw, kind in [('caixa','caixa'),('repique','repique'),('repinique','repique'),
                     ('timbau','timbau'),('tamborim','tamborim'),
                     ('chocalho','chocalho'),('agog','agogo')]:
        if kw in n:
            return kind
    return None

def _map(note, table):
    return table.get(note, '0')

def map_break(brk):
    result, unmapped = [], {}
    for name, notes in brk.tracks.items():
        kind = _classify(name)
        if kind is None:
            print(f'  ⚠️  Instrument not recognised, skipped: {name}')
            continue
        table_map = {
            'surdo_split': None, 'high_surdo': ('7', _HIGH_SURDO),
            'caixa': ('5', _CAIXA), 'repique': ('3', _REPIQUE),
            'timbau': ('6', _TIMBAU), 'tamborim': ('2', _TAMBORIM),
            'chocalho': ('1', _CHOCALHO), 'agogo': ('0', _AGOGO),
        }
        if kind == 'surdo_split':
            result.append(MappedTrack('9', [_map(n, _LOW_SURDO) for n in notes]))
            result.append(MappedTrack('8', [_map(n, _MID_SURDO) for n in notes]))
        else:
            iid, table = table_map[kind]
            mapped_notes = [_map(n, table) for n in notes]
            bad = {n for n in notes if n != '0' and n not in table}
            if bad:
                unmapped[name] = bad
            result.append(MappedTrack(iid, mapped_notes))
    for name, chars in unmapped.items():
        print(f'  ⚠️  Unmapped notes in {name}: {", ".join(sorted(chars))} → treated as rest')
    return result

# ── encode ─────────────────────────────────────────────────────────────────────

_B64 = '0123456789abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ~_'
_INSTRUMENT_BASE = {'0':3,'1':3,'2':3,'3':8,'5':5,'6':4,'7':3,'8':3,'9':3}

def _url_encode_number(n):
    if n == 0:
        return '0'
    out = []
    while n > 0:
        n, r = divmod(n, 64)
        out.append(_B64[r])
    return ''.join(reversed(out))

def encode_url(tracks, tempo=120, n_bars=None):
    steps = len(tracks[0].notes)
    if n_bars is None:
        n_bars = steps // 16
    parts = []
    for t in tracks:
        base = _INSTRUMENT_BASE[t.instrument_id]
        num = 0
        for d in t.notes:
            num = num * base + int(d)
        parts.append(t.instrument_id + _url_encode_number(num))
    return 'https://bananadrum.net/?a2=4-4.' + str(tempo) + '.' + str(n_bars) + '.1-4.16.' + '.'.join(parts)

print('✅ Setup complete — scroll down and fill in the form below.')

In [ ]:
#@title Generate BananaDrum link { run: "auto" }

sheets_url = "" #@param {type:"string", placeholder:"Paste the Google Sheets URL here"}
break_number = 1 #@param {type:"integer"}
tempo = 120 #@param {type:"integer"}

# ── run ────────────────────────────────────────────────────────────────────────

if not sheets_url.strip():
    print('Paste a Google Sheets URL in the field above, then run this cell.')
else:
    try:
        csv_text = fetch_csv(sheets_url)
    except Exception as e:
        print(f'❌ Could not fetch sheet: {e}')
        raise SystemExit

    breaks = parse_sheet(csv_text)
    if not breaks:
        print('❌ No breaks found in the sheet.')
        raise SystemExit

    if break_number < 1 or break_number > len(breaks):
        print(f'❌ Break {break_number} not found. This sheet has {len(breaks)} break(s): 1–{len(breaks)}.')
        raise SystemExit

    brk = breaks[break_number - 1]
    tracks = map_break(brk)

    if not tracks:
        print(f'❌ Break {break_number} "{brk.name}" has no recognised instruments.')
        raise SystemExit

    n_bars = max(len(t.notes) for t in tracks) // 16
    url = encode_url(tracks, tempo=tempo, n_bars=n_bars)

    print(f'Break {break_number}: {brk.name}')
    print(f'{n_bars} bars — {len(tracks)} instruments')
    print()
    print('🔗 ' + url)